# ***Data Ingestion***

In [ ]:
# import all libraries
import os
from dotenv import load_dotenv
load_dotenv()


# langchain
from langchain_text_splitters import TextSplitter, RecursiveCharacterTextSplitter
from langchain_community import document_loaders
from langchain_community.document_loaders import TextLoader

# Embedding Model
from langchain_community.embeddings import JinaEmbeddings




In [ ]:
groq_key= os.getenv("GROQ_API_KEY")

print('ENV VAR Loaded...')

In [ ]:
# Load the data
loader= TextLoader(r'D:\RAG Project\HR Policy\Data\Policy.txt', autodetect_encoding= True)
documents= loader.load()
print(documents)


## LANGCHAIN DOCUMENT

Langchain processes everything in form of documents

Documents 

Page Content -- the actual data.

METADATA -- Extra data about the data.


# ***Some NOTES***

- We have to know how much the length of our actual data. This helps us to how much chunk size is best for our data.

- 

In [ ]:
# See how much length of our data
print(f'Total characters in documents: {len(documents[0].page_content)}')


# ***Split our DATA***

In [ ]:
text_splitter= RecursiveCharacterTextSplitter(
    chunk_size= 300,
    chunk_overlap= 80
)

chunks= text_splitter.split_documents(documents)
print(chunks)


In [ ]:
# See the content each chunk contain
print(chunks[3].page_content)

# ***Embedding***

In [ ]:
from langchain_community.embeddings import JinaEmbeddings

jina_api= os.getenv('JINA_API_KEY')
embeddings_model = JinaEmbeddings(
    model_name="jina-embeddings-v2-base-en"
)

print('EMBED MODEL IS READY THE NAME IS: ', embeddings_model.model_name)

# ***Store Data in Vector Database***

In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
vector_store= FAISS.from_documents(chunks, embeddings_model)

# ***Tool***

In [ ]:
retriever= vector_store.as_retriever(search_kwargs= {'k': 3}) # return top 3 relevent chunks

def search_hr_policy(question: str) -> str:

    # This doc string is essential. This tells the llm what to do.
    """ 
        Search the HR Policy document for information about leave, work from home,
        porbation, notice period, reimbursement, code of conduct, holidays, or exit process

    """


    matching_chunks= retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

# ***Data Retrieval***

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv(override=True)


llm= ChatGroq(
    model= "openai/gpt-oss-20b",
    temperature= 0.7,  # this handles the llm's creativity. It is align between 0-1
    api_key= os.getenv("GROQ_API_KEY")
)

llm.model_name

In [ ]:
test_response= llm.invoke("Hey is learning RAG hard? ")
# this helps us to use the llm model.



In [ ]:
# Checks our api key perfectly works from env file or not


import os
from dotenv import load_dotenv

load_dotenv(override=True)  # Forces .env to overwrite cached system variables

key = os.getenv("GROQ_API_KEY")

if not key:
  print("❌ Error: GROQ_API_KEY is completely missing (None).")
else:
  print(f"✅ Key found! Length: {len(key)}")
  print(f"Starts with: {repr(key[:7])} | Ends with: {repr(key[-4:])}")


In [ ]:
print(test_response.content)

# ***AI Agent***

In [ ]:
from langchain.agents import create_agent

hr_assistant= create_agent(
    model= llm,
    tools= [search_hr_policy],
    system_prompt= """" 
        your are a friendly HR assistant.
        My org name is Sky Company Pvt Ltd.
        Always use the search_hr_policy tool to look up
        facts before answering
        If the answer isn't in the search results, say you don't know " 
        Instead of guessing."
    
    """
)

print('HR Assistant is ready to answer questions.')

In [ ]:
# Create a function that takes questions from the user

def ask_hr_assistant(question: str) -> str:
    """ Send a quesiton to the RAG agent and print a nicely formatted answer."""
    print('=' * 60)
    print("Question:", question)
    print('-' * 60)

    # Here we expect answers from our AI Agent not directly from the llm.
    response= hr_assistant.invoke({'messages': [{'role': 'user', 'content': question}]})
    answer= response['messages'][-1].content

    print('Answer:', answer)
    print('=' * 60)
    print()
    return answer

# All types of messages

- System Messages -> HR Assistant 

- Human Message -> Tell Me About Politics

- AI Message -> Hey These Are Thepolices


- **NOTE:** So the last messeage will be the ai message. Second last will be the Human Message and the first message will be the system message.

In [ ]:
question= 'What kind of things my company does?'


In [ ]:
# Response

response= hr_assistant.invoke(
    {
        "messages":[
            {
                'role': 'user',
                'content': question
            }
        ]
    }
)

In [ ]:
responsee= response['messages'][-1]
responsee.content